## 연습 1: TelConnect Unity Catalog 구조 설정하기

이 연습에서는 Lakeflow 작업(Job) 태스크가 읽고 쓸 Unity Catalog 기반을 준비합니다. 메달리온 아키텍처 패턴을 따르는 세 개의 스키마를 가진 카탈로그를 생성합니다:

- `bronze` — 수집된 원시 CDR 레코드
- `silver` — 정제 및 검증된 CDR 데이터
- `gold` — 보고용으로 집계된 네트워크 사용량 지표

### ✅ 제공됨: 처리 날짜에 대한 작업 매개변수 받기

이 노트북이 Lakeflow 작업 태스크로 실행될 때 작업은 `processing_date` 매개변수를 자동으로 전달합니다. `dbutils.widgets` API를 통해 해당 값을 노트북 내부에서 사용할 수 있습니다.

아래 셀을 실행하여 위젯을 등록하세요. 수동으로 실행할 때는 오늘 날짜가 기본값으로 사용됩니다.

In [ ]:
# ✅ 이 셀을 실행하세요 — 작업 매개변수 위젯을 등록합니다

import datetime

dbutils.widgets.text("processing_date", str(datetime.date.today()), "Processing Date (YYYY-MM-DD)")
processing_date = dbutils.widgets.get("processing_date")

print(f"Processing date: {processing_date}")

### ✅ 제공됨: Unity Catalog 구조 생성하기

아래 셀을 실행하여 `telconnect_lab` 카탈로그와 세 개의 메달리온 스키마를 생성하세요. 이 항목들은 실습 전반에 걸쳐 사용됩니다.


In [ ]:
# 기본 스토리지가 활성화되어 있지 않으므로, 기본 카탈로그의 루트에서 스토리지 경로를 상속받습니다.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_ws*")]
dplab11_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dplab11_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")

In [ ]:
# ✅ 이 셀을 실행하세요 — Unity Catalog 구조를 생성합니다

catalog = "telconnect_lab"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog} MANAGED LOCATION '{storage_root}' COMMENT 'TelConnect 네트워크 분석 플랫폼'")
print(f"✅ Catalog '{catalog}' created.")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze COMMENT '수집된 원시 통화 상세 기록(CDR)'")
print(f"✅ Schema '{catalog}.bronze' created.")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver COMMENT '정제 및 검증된 CDR 데이터'")
print(f"✅ Schema '{catalog}.silver' created.")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold COMMENT '보고용으로 집계된 네트워크 사용량 지표'")
print(f"✅ Schema '{catalog}.gold' created.")


## 연습 2: 원시 CDR 데이터 수집 (Bronze 계층)

Bronze 계층은 수신한 그대로의 원시 통화 상세 기록(CDR)을 저장합니다. 단일 처리 일자의 네트워크 이벤트를 나타내는 샘플 CSV가 제공됩니다. 이를 워크스페이스에 로드하고, Bronze Delta 테이블로 등록한 다음, 레코드 개수를 확인합니다.

### ✅ 제공됨: raw_uploads 볼륨 생성 및 샘플 데이터 다운로드

아래 셀을 실행하여 Unity Catalog 볼륨을 생성하고 GitHub에서 `telecom_cdrs.csv`를 볼륨으로 자동 다운로드하세요.


In [ ]:
# ✅ 이 셀을 실행하세요 — raw_uploads 볼륨을 생성하고 샘플 CDR 데이터를 다운로드합니다

import urllib.request

spark.sql("CREATE VOLUME IF NOT EXISTS telconnect_lab.bronze.raw_uploads COMMENT '원시 CDR 파일을 위한 랜딩 존'")
print("✅ Volume telconnect_lab.bronze.raw_uploads is ready.")

url = "https://raw.githubusercontent.com/asddai/AzureDatabricks/main/Labs/data/telecom_cdrs.csv"
dest = "/Volumes/telconnect_lab/bronze/raw_uploads/telecom_cdrs.csv"
urllib.request.urlretrieve(url, dest)
print(f"✅ telecom_cdrs.csv downloaded to {dest}")


### ✅ 제공됨: CDR CSV를 DataFrame으로 로드하기

아래 셀을 실행하여 업로드된 CSV를 Spark DataFrame으로 읽어들이세요. 출력을 확인하면서 연습 3에서 정제할 품질 문제가 있는 레코드가 무엇인지 살펴보세요.


In [ ]:
# ✅ 이 셀을 실행하세요 — 볼륨에서 원시 CDR CSV를 로드합니다

from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

cdr_schema = StructType([
    StructField("call_id",           StringType(),  True),
    StructField("caller_msisdn",     StringType(),  True),
    StructField("callee_msisdn",     StringType(),  True),
    StructField("call_start_time",   StringType(),  True),
    StructField("call_end_time",     StringType(),  True),
    StructField("duration_seconds",  IntegerType(), True),
    StructField("call_type",         StringType(),  True),
    StructField("call_status",       StringType(),  True),
    StructField("network_type",      StringType(),  True),
    StructField("cell_tower_id",     StringType(),  True),
    StructField("region",            StringType(),  True),
    StructField("roaming",           StringType(),  True),
    StructField("data_mb",           DoubleType(),  True),
    StructField("sms_count",         IntegerType(), True),
])

raw_df = (
    spark.read
    .option("header", True)
    .schema(cdr_schema)
    .csv("/Volumes/telconnect_lab/bronze/raw_uploads/telecom_cdrs.csv")
)

raw_df.display()
print(f"\nRaw CDR records loaded: {raw_df.count()}")


### ✅ 제공됨: 원시 CDR 데이터를 Bronze Delta 테이블에 쓰기

아래 셀을 실행하여 `raw_df`를 Unity Catalog의 관리형 Delta 테이블로 영구 저장하세요. 이는 이후의 모든 정제 및 집계 작업의 시작점입니다.


In [ ]:
# ✅ 이 셀을 실행하세요 — 원시 CDR 데이터를 Bronze Delta 테이블에 씁니다

raw_df.write.format("delta").mode("overwrite").saveAsTable("telconnect_lab.bronze.cdr_bronze")

count = spark.table("telconnect_lab.bronze.cdr_bronze").count()
print(f"✅ Bronze table written. Record count: {count}")


## 연습 3: CDR 데이터 정제 및 검증 (Silver 계층)

Silver 계층에는 검증을 통과한 CDR이 포함됩니다. 운영 파이프라인에서는 정제된 레코드만 집계 및 청구 시스템으로 다운스트림 전달됩니다.

원시 데이터를 주의 깊게 검토하세요. 다음과 같이 수정해야 할 품질 문제를 발견하게 됩니다:

| 문제 | 예시 | 처리 규칙 |
|---|---|---|
| 음수 통화 시간 | CDR010 | `duration_seconds < 0`인 레코드 삭제 |
| VOICE/DATA 통화의 수신자 누락 | CDR007 | `callee_msisdn`이 null인 VOICE 및 DATA 레코드 삭제 |
| 통화 시간이 0인 완료된 통화 | 여러 건 | `call_status = COMPLETED`이고 `duration_seconds = 0`인 VOICE 레코드 삭제 |
| 유효하지 않은 call_type 값 | 데이터 확인 | 다음 값만 유지: `VOICE`, `DATA`, `SMS` |

In [ ]:
# ✅ 이 셀을 실행하세요 — 연습 2에서 생성한 bronze 테이블에서 읽어옵니다

bronze_df = spark.table("telconnect_lab.bronze.cdr_bronze")
print(f"Bronze record count: {bronze_df.count()}")
bronze_df.display()

### ✅ 제공됨: 정제 규칙 적용 및 Silver에 쓰기

아래 셀을 실행하세요. `bronze_df`에 네 가지 데이터 품질 규칙을 적용하고 검증된 레코드를 Silver 테이블에 씁니다. 각 단계에서 몇 개의 레코드가 삭제되었는지 출력 내용을 확인하세요.


In [ ]:
# ✅ 이 셀을 실행하세요 — CDR 정제 규칙을 적용하고 Silver에 씁니다

from pyspark.sql.functions import col

bronze_count = bronze_df.count()

silver_df = (
    bronze_df
    # 규칙 1: 음수 통화 시간 삭제
    .filter(col("duration_seconds") >= 0)
    # 규칙 2: 수신자가 없는 VOICE 및 DATA 레코드 삭제
    .filter(
        ~((col("call_type").isin("VOICE", "DATA")) & col("callee_msisdn").isNull())
    )
    # 규칙 3: 통화 시간이 0인 완료된 VOICE 통화 삭제
    .filter(
        ~((col("call_type") == "VOICE") & (col("call_status") == "COMPLETED") & (col("duration_seconds") == 0))
    )
    # 규칙 4: 유효한 통화 유형만 유지
    .filter(col("call_type").isin("VOICE", "DATA", "SMS"))
)

silver_df.write.format("delta").mode("overwrite").saveAsTable("telconnect_lab.silver.cdr_silver")

silver_count = spark.table("telconnect_lab.silver.cdr_silver").count()
print(f"✅ Silver table written.")
print(f"   Bronze records : {bronze_count}")
print(f"   Silver records : {silver_count}")
print(f"   Records dropped: {bronze_count - silver_count}")


## 연습 4: 네트워크 지표 집계 (Gold 계층)

Gold 계층에는 대시보드와 운영 팀이 사용하는 집계된 지표가 포함됩니다. TelConnect의 네트워크 운영 센터는 지역 및 네트워크 유형별로 다음을 보여주는 일일 요약이 필요합니다:

- 처리된 총 통화 수
- 성공한 총 통화 수 (`COMPLETED` 상태)
- 통화 끊김률 (`DROPPED`된 통화의 비율)
- 평균 통화 시간 (`VOICE` 통화만 해당)
- 전송된 총 데이터(MB) (`DATA` 통화만 해당)

In [ ]:
# ✅ 이 셀을 실행하세요 — silver 테이블에서 읽어옵니다

silver_df = spark.table("telconnect_lab.silver.cdr_silver")
print(f"Silver record count: {silver_df.count()}")

### ✅ 제공됨: 지역별 네트워크 요약 만들기

아래 셀을 실행하세요. Silver CDR 데이터를 지역 및 네트워크 유형별로 집계하여 다섯 가지 운영 지표를 계산하고 그 결과를 Gold 테이블에 씁니다.


In [ ]:
# ✅ 이 셀을 실행하세요 — CDR Silver 데이터를 Gold 네트워크 요약으로 집계합니다

from pyspark.sql.functions import (
    count, sum as spark_sum, avg, when, round as spark_round
)

gold_df = (
    silver_df
    .groupBy("region", "network_type")
    .agg(
        count("*").alias("total_calls"),
        count(when(col("call_status") == "COMPLETED", 1)).alias("completed_calls"),
        spark_round(
            count(when(col("call_status") == "DROPPED", 1)) * 100.0 / count("*"), 2
        ).alias("drop_rate_pct"),
        spark_round(
            avg(when((col("call_type") == "VOICE") & (col("duration_seconds") > 0), col("duration_seconds"))), 1
        ).alias("avg_voice_duration_sec"),
        spark_round(
            spark_sum(when(col("call_type") == "DATA", col("data_mb"))), 2
        ).alias("total_data_mb"),
    )
    .orderBy("region", "network_type")
)

gold_df.write.format("delta").mode("overwrite").saveAsTable("telconnect_lab.gold.network_summary")

print(f"✅ Gold table written. Rows: {spark.table('telconnect_lab.gold.network_summary').count()}")
spark.table("telconnect_lab.gold.network_summary").display()


## 연습 5: 노트북을 작업 실행 준비 상태로 만들기

이 노트북이 Lakeflow 작업 태스크로 실행될 때는 결과를 작업에 다시 전달해야 합니다. Databricks는 두 가지 메커니즘을 제공합니다:

- **`dbutils.notebook.exit()`** — 노트북을 종료하고 `Run Notebook` 태스크 유형을 통해 이 노트북을 호출하는 모든 태스크에 문자열 값을 전달합니다.
- **`dbutils.jobs.taskValues.set()`** — 동일한 작업 실행 내의 다운스트림 태스크가 검색할 수 있도록 이름이 지정된 값을 저장합니다.

여기서는 노트북을 관찰 가능하고 연결 가능하게 만들기 위해 두 가지를 모두 사용합니다.

### ✅ 제공됨: Gold 레코드 수를 세고 결과를 작업과 공유하기

아래 셀을 실행하세요. Gold 행 수를 태스크 값으로 게시하여(다운스트림 작업 태스크가 읽을 수 있도록) 구조화된 JSON 상태 문자열과 함께 노트북을 종료합니다. 이것이 노트북을 적절한 Lakeflow 작업 태스크로 동작하게 만드는 요소입니다.


In [ ]:
# ✅ 이 셀을 실행하세요 — 결과를 게시하고 노트북을 작업 태스크로 종료합니다

import json

gold_row_count = spark.table("telconnect_lab.gold.network_summary").count()

# 동일한 작업 실행 내의 다운스트림 태스크와 레코드 수 공유
dbutils.jobs.taskValues.set(key="gold_row_count", value=gold_row_count)

print(f"✅ Pipeline complete for processing_date={processing_date}. Gold rows: {gold_row_count}")

# 구조화된 상태와 함께 종료 — 작업 실행 타임라인에서 읽을 수 있음
result = json.dumps({"status": "success", "processing_date": processing_date, "gold_rows": gold_row_count})
dbutils.notebook.exit(result)
